# Experiment 12: Face Mask Detection Using CNN

Upload/extract the dataset in `/content/face_mask_dataset/` so it contains two class folders, such as `with_mask/` and `without_mask/`. Then run the single code cell below. Change `DATASET_DIR` in the code if needed.

In [ ]:
# EXPERIMENT 12: FACE MASK DETECTION USING CNN
# Google Colab: Runtime > Change runtime type > (GPU optional)
#
# Dataset expected in this structure after downloading and extracting:
# face_mask_dataset/
#   WithMask/   (images of people wearing masks)
#   WithoutMask/ (images of people not wearing masks)
#
# This notebook code can also download a ZIP dataset if you provide its direct URL.
# For a Kaggle dataset, download it from Kaggle and upload/extract the ZIP in Colab,
# then set DATASET_DIR to the folder containing the two class folders.

import os, random, shutil, zipfile, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 1. DATASET PATH: update this to your extracted dataset folder in Google Colab.
# Example: "/content/face_mask_dataset"
DATASET_DIR = "/content/face_mask_dataset"

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 12  # Increase to 20-30 if time and compute permit.
AUTOTUNE = tf.data.AUTOTUNE
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# ---- 2. FIND THE TWO CLASS FOLDERS ----
# Supports folders named WithMask/WithoutMask, mask/no_mask, masked/unmasked, etc.
def find_class_dirs(root):
    root = pathlib.Path(root)
    if not root.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {root}\n"
            "Download/extract a face-mask image dataset in Colab and set DATASET_DIR."
        )
    candidates = [p for p in root.iterdir() if p.is_dir()]
    def norm(s):
        return "".join(ch.lower() for ch in s if ch.isalnum())
    mask_names = {"withmask", "masked", "mask", "wearingmask", "withfacemask"}
    no_mask_names = {"withoutmask", "unmasked", "nomask", "withoutfacemask", "notwearingmask"}
    mask_dir = next((p for p in candidates if norm(p.name) in mask_names), None)
    no_mask_dir = next((p for p in candidates if norm(p.name) in no_mask_names), None)
    if mask_dir and no_mask_dir:
        return [mask_dir, no_mask_dir]
    # Fallback: accept exactly two class directories, in alphabetical order.
    if len(candidates) == 2:
        return sorted(candidates, key=lambda p: p.name.lower())
    raise ValueError(
        "Could not identify two class folders. Ensure DATASET_DIR directly contains "
        "one folder for masked faces and one for unmasked faces. Found: "
        + ", ".join(p.name for p in candidates)
    )

class_dirs = find_class_dirs(DATASET_DIR)
class_names = [p.name for p in class_dirs]
print("TensorFlow version:", tf.__version__)
print("Dataset directory:", DATASET_DIR)
print("Class names:", class_names)

# ---- 3. BASIC DATA DESCRIPTION / FILE AUDIT ----
records = []
for label, folder in enumerate(class_dirs):
    for path in folder.rglob("*"):
        if path.is_file() and path.suffix.lower() in VALID_EXTS:
            records.append({"path": str(path), "label": label,
                            "class_name": class_names[label],
                            "extension": path.suffix.lower(),
                            "file_size_kb": round(path.stat().st_size / 1024, 2)})

df = pd.DataFrame(records)
if df.empty:
    raise ValueError("No images found. Check the folder path and image file extensions.")

print("\nFirst five records:")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nData types and non-null counts:")
print(df.info())
print("\nDescriptive statistics:")
print(df[["label", "file_size_kb"]].describe())
print("\nClass counts:")
print(df["class_name"].value_counts())

# Remove duplicate file paths and exact duplicate files using SHA-256.
# Exact file hashing avoids relying only on filenames.
import hashlib
def sha256_file(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

df["sha256"] = df["path"].map(sha256_file)
before = len(df)
df = df.drop_duplicates(subset="sha256").reset_index(drop=True)
print(f"\nExact duplicate image files removed: {before - len(df)}")

# Validate that images can be opened and decoded; remove corrupt images.
valid_rows = []
bad_paths = []
for row in df.itertuples(index=False):
    try:
        raw = tf.io.read_file(row.path)
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        _ = img.shape
        valid_rows.append({"path": row.path, "label": row.label,
                           "class_name": row.class_name,
                           "extension": row.extension,
                           "file_size_kb": row.file_size_kb,
                           "sha256": row.sha256})
    except Exception:
        bad_paths.append(row.path)
df = pd.DataFrame(valid_rows).reset_index(drop=True)
print("Corrupt/unreadable images removed:", len(bad_paths))
if df.empty:
    raise ValueError("No valid images remained after validation.")
if df["label"].nunique() != 2:
    raise ValueError("Both classes must contain valid images.")

# ---- 4. EXPLORATORY DATA ANALYSIS (EDA) ----
plt.figure(figsize=(6, 4))
df["class_name"].value_counts().reindex(class_names).plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Display sample images.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row_i, class_name in enumerate(class_names):
    samples = df[df["class_name"] == class_name].sample(
        n=min(4, (df["class_name"] == class_name).sum()), random_state=SEED
    )
    for col_i, rec in enumerate(samples.itertuples(index=False)):
        img = tf.keras.utils.load_img(rec.path, target_size=IMG_SIZE)
        axes[row_i, col_i].imshow(img)
        axes[row_i, col_i].set_title(class_name)
        axes[row_i, col_i].axis("off")
    for col_i in range(len(samples), 4):
        axes[row_i, col_i].axis("off")
plt.suptitle("Example Images from Each Class")
plt.tight_layout()
plt.show()

# ---- 5. TRAIN / VALIDATION / TEST SPLIT ----
# Stratified split keeps class proportions similar in each set.
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)
print("\nSplit sizes:")
print(f"Train: {len(train_df)} ({len(train_df)/len(df):.1%})")
print(f"Validation: {len(val_df)} ({len(val_df)/len(df):.1%})")
print(f"Test: {len(test_df)} ({len(test_df)/len(df):.1%})")

# Use integer labels 0 and 1; pixel scaling to [0,1] happens in the input pipeline.
# Resize all images to a fixed size; this is the key preprocessing step.
def make_dataset(frame, training=False):
    paths = frame["path"].to_numpy()
    labels = frame["label"].astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_image(path, label):
        image_bytes = tf.io.read_file(path)
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

# Check class imbalance. Class weights are computed from training labels only.
unique_labels = np.array(sorted(train_df["label"].unique()))
weights = compute_class_weight(
    class_weight="balanced", classes=unique_labels, y=train_df["label"].to_numpy()
)
class_weights = {int(label): float(weight) for label, weight in zip(unique_labels, weights)}
print("\nClass weights (used to reduce imbalance impact):", class_weights)

# ---- 6. DEFINE CNN VARIANTS: 1, 2, AND 3 CONVOLUTIONAL LAYERS ----
# All models use the same split, image size, optimizer, epochs, and evaluation metrics.
def build_cnn(num_conv_layers):
    model = tf.keras.Sequential(name=f"CNN_{num_conv_layers}_conv_layers")
    model.add(tf.keras.layers.Input(shape=(*IMG_SIZE, 3)))
    filters = [32, 64, 128]
    for i in range(num_conv_layers):
        model.add(tf.keras.layers.Conv2D(filters[i], (3, 3), padding="same",
                                         activation="relu"))
        model.add(tf.keras.layers.MaxPooling2D((2, 2)))
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(64, activation="relu"))
    model.add(tf.keras.layers.Dropout(0.30))
    model.add(tf.keras.layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Train and compare models.
results = []
histories = {}
models = {}
test_labels = test_df["label"].to_numpy()

for n_layers in [1, 2, 3]:
    print("\n" + "=" * 65)
    print(f"Training CNN with {n_layers} convolutional layer(s)")
    model = build_cnn(n_layers)
    model.summary()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
    ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )
    probabilities = model.predict(test_ds, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)

    # Binary metrics; class 1 is the second class in class_names.
    results.append({
        "Model": f"CNN ({n_layers} conv layer{'s' if n_layers > 1 else ''})",
        "Conv_Layers": n_layers,
        "Test_Accuracy": accuracy_score(test_labels, predictions),
        "Precision_macro": precision_score(test_labels, predictions, average="macro", zero_division=0),
        "Recall_macro": recall_score(test_labels, predictions, average="macro", zero_division=0),
        "F1_macro": f1_score(test_labels, predictions, average="macro", zero_division=0),
        "Epochs_Trained": len(history.history["loss"])
    })
    histories[n_layers] = history
    models[n_layers] = model

results_df = pd.DataFrame(results).sort_values(
    by=["F1_macro", "Test_Accuracy"], ascending=False
).reset_index(drop=True)
print("\nMODEL COMPARISON (actual values from this run):")
print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---- 7. PERFORMANCE VISUALIZATION ----
plt.figure(figsize=(8, 4))
for n_layers, history in histories.items():
    plt.plot(history.history["val_accuracy"], label=f"{n_layers} conv layer(s)")
plt.title("Validation Accuracy by CNN Depth")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
for n_layers, history in histories.items():
    plt.plot(history.history["val_loss"], label=f"{n_layers} conv layer(s)")
plt.title("Validation Loss by CNN Depth")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.legend()
plt.tight_layout()
plt.show()

# ---- 8. FINAL TEST REPORT FOR THE BEST MODEL ----
best_layers = int(results_df.iloc[0]["Conv_Layers"])
best_model = models[best_layers]
best_probs = best_model.predict(test_ds, verbose=0).ravel()
best_preds = (best_probs >= 0.5).astype(int)

print(f"\nBest model selected by macro-F1, then accuracy: {best_layers} convolutional layer(s)")
print("\nClassification report:")
print(classification_report(
    test_labels, best_preds, labels=[0, 1], target_names=class_names, zero_division=0
))

cm = confusion_matrix(test_labels, best_preds, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
    cmap="Blues", values_format="d"
)
plt.title(f"Confusion Matrix — CNN with {best_layers} Conv Layer(s)")
plt.tight_layout()
plt.show()

# ---- 9. SAMPLE PREDICTIONS ----
sample_df = test_df.sample(n=min(8, len(test_df)), random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i >= len(sample_df):
        ax.axis("off")
        continue
    rec = sample_df.iloc[i]
    image = tf.keras.utils.load_img(rec["path"], target_size=IMG_SIZE)
    x = tf.keras.utils.img_to_array(image) / 255.0
    probability = float(best_model.predict(np.expand_dims(x, 0), verbose=0)[0][0])
    pred_label = int(probability >= 0.5)
    ax.imshow(image)
    ax.set_title(f"Actual: {class_names[int(rec['label'])]}\n"
                 f"Predicted: {class_names[pred_label]}")
    ax.axis("off")
plt.suptitle("Sample Test Predictions")
plt.tight_layout()
plt.show()

# ---- 10. SAVE RESULTS ----
results_df.to_csv("/content/cnn_model_comparison.csv", index=False)
best_model.save("/content/best_face_mask_cnn.keras")
print("\nSaved: /content/cnn_model_comparison.csv")
print("Saved: /content/best_face_mask_cnn.keras")
print("\nIMPORTANT: Use the metrics printed above in your report. Do not invent or pre-fill results.")
